# 01 — Exploratory Data Analysis
Understand the raw datasets before any modelling.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#050a0f'
matplotlib.rcParams['axes.facecolor']   = '#0a1520'
matplotlib.rcParams['text.color']       = '#d0e8e0'
matplotlib.rcParams['axes.labelcolor']  = '#7a9e90'
matplotlib.rcParams['xtick.color']      = '#3a5e50'
matplotlib.rcParams['ytick.color']      = '#3a5e50'
print("Imports OK")

## 1. Load raw data

In [ ]:
from src.data_loader import load_jhu_cases, load_jhu_deaths, load_owid

cases  = load_jhu_cases(download=True)
deaths = load_jhu_deaths(download=True)
owid   = load_owid(download=True)

print("Cases shape: ", cases.shape)
print("Deaths shape:", deaths.shape)
print("OWID shape:  ", owid.shape)
cases.head()

## 2. Date range & country count

In [ ]:
print(f"Date range: {cases.date.min()} → {cases.date.max()}")
print(f"Countries : {cases.country.nunique()}")

## 3. Top 10 countries by total cases

In [ ]:
top10 = (cases.groupby('country')['confirmed_cumulative']
         .max().sort_values(ascending=False).head(10))

fig, ax = plt.subplots(figsize=(12, 5))
top10.plot(kind='barh', ax=ax, color='#00c896')
ax.set_title('Top 10 Countries by Total Confirmed Cases', color='#d0e8e0')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Time-series for selected countries

In [ ]:
countries = ['United States','India','Brazil','United Kingdom','Germany']
fig, ax = plt.subplots(figsize=(14, 5))
colors = ['#00c896','#38b6ff','#ffb830','#ff4d6d','#9b6dff']

for c, col in zip(countries, colors):
    sub = cases[cases.country == c].sort_values('date')
    # daily new
    sub = sub.copy()
    sub['new'] = sub['confirmed_cumulative'].diff().clip(lower=0)
    sub['new7'] = sub['new'].rolling(7).mean()
    ax.plot(sub['date'], sub['new7'], label=c, color=col, linewidth=1.5)

ax.set_title('7-day Average New Cases — Selected Countries', color='#d0e8e0')
ax.legend(facecolor='#0d1b2a', edgecolor='#1a3040', labelcolor='#7a9e90')
plt.tight_layout()
plt.show()

## 5. Missing value analysis (OWID)

In [ ]:
missing = (owid.isnull().mean() * 100).sort_values(ascending=False)
print(missing[missing > 0].to_string())

## 6. Correlation heatmap (OWID numeric features)

In [ ]:
import seaborn as sns

# Take latest row per country
latest = owid.sort_values('date').groupby('country').last().reset_index()
num_cols = latest.select_dtypes(include='number').columns.tolist()
corr = latest[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, ax=ax, cmap='RdYlGn', center=0,
            linewidths=0.3, linecolor='#0d1b2a',
            annot=True, fmt='.2f', annot_kws={'size': 7})
ax.set_title('OWID Feature Correlation', color='#d0e8e0')
plt.tight_layout()
plt.show()